In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Generate x values
np.random.seed(12345)
x = np.linspace(0, 10, 500)

# Generate three highly non-linear curves
y1 = np.sin(x) + 0.3 * np.cos(3 * x)
y2 = np.exp(-0.2 * x) * np.sin(2 * x) + 0.2 * np.random.randn(len(x))
y3 = np.tanh(x - 5) + 0.1 * np.sin(5 * x)


# Randomly choose 4 non-overlapping intervals on the x axis
interval_edges = []
used_indices = set()

while len(interval_edges) < 5:
    interval_width = np.random.randint(5, 10) / 10  # width of each interval

    # pick a random center index, avoiding edges
    center_idx = np.random.randint(50, len(x) - 50)
    left = max(x[0], x[center_idx] - interval_width / 2)
    right = min(x[-1], x[center_idx] + interval_width / 2)

    # check for overlap with existing intervals
    overlap = False

    for le, ri in interval_edges:
        if not (right <= le or left >= ri):
            overlap = True
            break

    if not overlap:
        interval_edges.append((left, right))

interval_edges = sorted(interval_edges)

fig = go.Figure()

fig.add_trace(go.Scatter(x=x, y=y1, mode="lines", name="Curve 1"))
fig.add_trace(go.Scatter(x=x, y=y2, mode="lines", name="Curve 2"))
fig.add_trace(go.Scatter(x=x, y=y3, mode="lines", name="Curve 3"))

for left, right in interval_edges:
    fig.add_vrect(x0=left, x1=right, fillcolor="lightgrey", opacity=0.5, line_width=0)
    fig.add_vline(x=left, line={"color": "black", "dash": "dash", "width": 1})
    fig.add_vline(x=right, line={"color": "black", "dash": "dash", "width": 1})

fig.update_layout(
    title="Non-linear Curves with Decision Tree Intervals",
    xaxis_title="x",
    yaxis_title="y",
    legend_title="Curves",
    width=900,
    height=500,
)

fig.show()


# Compute statistics for each interval and curve
stats = []

for i in range(len(interval_edges)):
    left, right = interval_edges[i]
    mask = (x >= left) & (x < right)
    for curve_idx, y in enumerate([y1, y2, y3], start=1):
        y_interval = y[mask]
        x_interval = x[mask]

        # Slope: linear fit
        if len(x_interval) > 1:
            slope = np.polyfit(x_interval, y_interval, 1)[0]
        else:
            slope = np.nan

        stats.append(
            {
                "Interval": f"{left:.2f} - {right:.2f}",
                "Curve": f"Curve {curve_idx}",
                "Mean": f"{np.mean(y_interval):.2f}",
                "Std": f"{np.std(y_interval):.2f}",
                "Slope": f"{slope:.2f}",
            }
        )

df_stats = pd.DataFrame(stats)
print(df_stats)

       Interval    Curve   Mean   Std  Slope
0   1.35 - 2.05  Curve 1   1.07  0.13   0.61
1   1.35 - 2.05  Curve 2  -0.14  0.30  -1.17
2   1.35 - 2.05  Curve 3  -0.95  0.05  -0.22
3   2.48 - 2.98  Curve 1   0.31  0.25  -1.72
4   2.48 - 2.98  Curve 2  -0.41  0.21   1.02
5   2.48 - 2.98  Curve 3  -0.91  0.04   0.25
6   5.00 - 5.90  Curve 1  -0.89  0.30   1.11
7   5.00 - 5.90  Curve 2  -0.35  0.19  -0.04
8   5.00 - 5.90  Curve 3   0.43  0.18   0.65
9   6.37 - 6.97  Curve 1   0.48  0.03   0.15
10  6.37 - 6.97  Curve 2   0.20  0.21   0.28
11  6.37 - 6.97  Curve 3   0.99  0.03  -0.01
12  7.71 - 8.41  Curve 1   1.10  0.10   0.46
13  7.71 - 8.41  Curve 2  -0.04  0.26  -0.67
14  7.71 - 8.41  Curve 3   1.03  0.06  -0.30


In [2]:
interval_edges

[(np.float64(1.3534068136272546), np.float64(2.0534068136272543)),
 (np.float64(2.4754509018036073), np.float64(2.9754509018036073)),
 (np.float64(5.000901803607214), np.float64(5.900901803607215)),
 (np.float64(6.3733466933867735), np.float64(6.973346693386773)),
 (np.float64(7.706112224448898), np.float64(8.406112224448897))]